In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Always use Delta format in Gold layer
GOLD_DB = "olist.gold"

# Create Gold database if it doesn't exist
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

In [0]:
# =============================================================
# CELL 1 — LOAD SILVER TABLES
# =============================================================

orders      = spark.table("olist.silver.olist_order")
order_items = spark.table("olist.silver.olist_order_item")
customers   = spark.table("olist.silver.olist_customer")
products    = spark.table("olist.silver.olist_product")
sellers     = spark.table("olist.silver.seller")
payments    = spark.table("olist.silver.olist_payment")
reviews     = spark.table("olist.silver.olist_reviews")
category    = spark.table("olist.silver.olist_product_category")
geolocation = spark.table("olist.silver.olist_geo")



In [0]:
# dim_orders
dim_orders =(
    orders.select("order_id","customer_id","order_status","order_purchase_timestamp","order_approved_at","order_delivered_carrier_date","order_delivered_customer_date","order_estimated_delivery_date")
    .withColumn("order_key", F.monotonically_increasing_id())
)

dim_orders.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_DB}.dim_orders")

In [0]:
# dimloaction

dim_geolocation = (
    geolocation
    .select("geolocation_city",
            "geolocation_latitude",
            "longitude",
            "geolocation_zip_code_prefix",
            "geolocation_state")
    .withColumn("geolocation_key", F.monotonically_increasing_id())
)

dim_geolocation.write.mode("overwrite").format("delta").saveAsTable(f"{GOLD_DB}.dim_geolocation")
dim_geolocation.show()

In [0]:
# dim date


# Pull all unique dates from order timestamps
date_df = (
    orders
    .select(F.to_date("order_purchase_timestamp").alias("full_date"))
    .distinct()
    .filter(F.col("full_date").isNotNull())
)

dim_date = (
    date_df
    .withColumn("date_key",        F.date_format("full_date", "yyyyMMdd").cast("int"))  # PK: 20180101
    .withColumn("year",            F.year("full_date"))
    .withColumn("quarter",         F.quarter("full_date"))
    .withColumn("month",           F.month("full_date"))
    .withColumn("month_name",      F.date_format("full_date", "MMMM"))
    .withColumn("week_of_year",    F.weekofyear("full_date"))
    .withColumn("day_of_month",    F.dayofmonth("full_date"))
    .withColumn("day_of_week",     F.dayofweek("full_date"))          # 1=Sun, 7=Sat
    .withColumn("day_name",        F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend",      F.dayofweek("full_date").isin([1, 7]))
    .withColumn("is_month_start",  F.dayofmonth("full_date") == 1)
    .withColumn("is_month_end",    F.col("full_date") == F.last_day("full_date"))
)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.dim_date")
)
print("dim_date saved")

In [0]:
# dim_customer

dim_customer = (
    customers
    .select(
        "customer_id",           # natural key (from source)
        "customer_unique_id",    # real unique customer identifier
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    )
    .dropDuplicates(["customer_id"])
    # Surrogate key: monotonically_increasing_id() is Spark's industry-standard way
    .withColumn("customer_key", F.monotonically_increasing_id())
)

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.dim_customer")
)
print(" dim_customer saved")

In [0]:
# dim_product 
dim_product = (
    products
    .select(
        "product_id",
        "product_category_name",
        "product_name_length",
        "product_description_length",
        "product_photos_qty",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    )
    .dropDuplicates(["product_id"])
    .withColumn("product_key", F.monotonically_increasing_id())
)

(
    dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.dim_product")
)
print("dim_product saved")

In [0]:
# dim_seller

dim_seller = (
    sellers
    .select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    )
    .dropDuplicates(["seller_id"])
    .withColumn("seller_key", F.monotonically_increasing_id())
)

(
    dim_seller.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.dim_seller")
)
print("dim_seller saved")

In [0]:
# dim_payment

dim_payment = (
    payments
    .select("order_id","payment_sequential","payment_type","payment_value","payment_installments")
    .withColumn("payment_key", F.monotonically_increasing_id())
)

(
    dim_payment.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.dim_payment")
)
print("dim_payment saved")

In [0]:
reviews.display()

In [0]:
# creating dim_review table


dim_review = (
    reviews
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp\r"
    )
    .dropDuplicates(["review_id"])
    .withColumn("review_key", F.monotonically_increasing_id())
)

(
    dim_review.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD_DB}.dim_review")
)
print(" dim_review saved")

In [0]:
# creating fact table

dim_customer_gold = spark.table(f"{GOLD_DB}.dim_customer")
dim_date_gold     = spark.table(f"{GOLD_DB}.dim_date")
dim_geolocation_gold =spark.table(f"{GOLD_DB}.dim_geolocation")
dim_payment_gold = spark.table(f"{GOLD_DB}.dim_payment")
dim_product_gold = spark.table(f"{GOLD_DB}.dim_product")
dim_review_gold = spark.table(f"{GOLD_DB}.dim_review")
dim_seller_gold = spark.table(f"{GOLD_DB}.dim_seller")
dim_orders_gold = spark.table(f"{GOLD_DB}.dim_orders")

fact_order_items = order_items.join(dim_orders_gold, on ="order_id", how ="left")\
                              .join(dim_customer_gold, on="customer_id", how ="left")\
                              .join(dim_product_gold, on ="product_id", how ="left")\
                              .join(dim_seller_gold, on = "seller_id", how = "left") \
                              .join(dim_payment_gold,  on="order_id",     how="left") \
                                  .join(dim_review_gold,   on="order_id",     how="left")\
                                      .withColumn("date_key",
                                    F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))\
                                         .withColumn("gross_revenue",F.col("price") + F.col("freight_value"))\
                                         .withColumn("delivery_days",F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))\
    .withColumn("estimated_vs_actual_days",
        F.datediff("order_delivered_customer_date", "order_estimated_delivery_date"))\
    .withColumn("is_late_delivery",
        F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"))
    

(
    fact_order_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("order_status")
    .saveAsTable(f"{GOLD_DB}.fact_order_items")
)
